# E-Commerce LTV & Attribution Analysis

Probabilistic Customer Lifetime Value (BG/NBD + Gamma-Gamma) on synthetic multi-category e-commerce data.

Outputs:
- LTV distribution and concentration
- Next-euro ranking by acquisition channel
- Top customers
- Cohort-style retention signals

Run all cells to generate the plots.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_data
from datetime import datetime, timedelta

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
np.random.seed(42)

In [ ]:
def generate_synthetic_ecommerce_data(n_customers=2500, days=540):
    markets = ['DE', 'FR', 'IT', 'ES', 'NL', 'BE', 'AT', 'Other']
    channels = ['Meta', 'Google', 'Organic', 'Email', 'Influencer', 'Direct']
    categories = ['Electronics', 'Fashion', 'Home', 'Beauty', 'Sports', 'Books']

    rows = []
    start = datetime(2024, 6, 1)

    for cid in range(1, n_customers + 1):
        market = np.random.choice(markets, p=[0.28, 0.18, 0.12, 0.10, 0.08, 0.07, 0.05, 0.12])
        channel = np.random.choice(channels, p=[0.30, 0.22, 0.18, 0.12, 0.10, 0.08])
        loyal = np.random.random() < 0.40
        n_orders = min(np.random.poisson(1.8 if loyal else 0.9) + 1, 12)

        first_order = start + timedelta(days=int(np.random.exponential(120)))
        if first_order > start + timedelta(days=days):
            continue

        for o in range(n_orders):
            order_date = first_order + timedelta(days=int(np.random.exponential(45) * (o + 1)))
            if order_date > start + timedelta(days=days):
                break
            n_items = np.random.choice([1, 2, 3, 4, 5], 
                p=[0.20, 0.35, 0.25, 0.12, 0.08] if loyal else [0.50, 0.30, 0.12, 0.05, 0.03])
            for _ in range(n_items):
                category = np.random.choice(categories)
                price = round(np.random.uniform(15, 120) * np.random.uniform(0.85, 1.25), 2)
                rows.append({
                    'customer_id': cid,
                    'order_date': order_date,
                    'category': category,
                    'price': price,
                    'market': market,
                    'channel': channel,
                    'is_loyal': loyal
                })

    return pd.DataFrame(rows)

df = generate_synthetic_ecommerce_data()
print(f'Orders: {len(df):,} | Customers: {df.customer_id.nunique():,}')
df.head()

In [ ]:
summary = summary_data_from_transaction_data(
    df, 'customer_id', 'order_date', monetary_value_col='price', freq='D'
)
summary = summary[summary['monetary_value'] > 0].copy()

bgf = BetaGeoFitter(penalizer_coef=0.01)
bgf.fit(summary['frequency'], summary['recency'], summary['T'])

returning = summary[summary['frequency'] > 0]
ggf = GammaGammaFitter(penalizer_coef=0.01)
ggf.fit(returning['frequency'], returning['monetary_value'])

summary['clv_12m'] = ggf.customer_lifetime_value(
    bgf, summary['frequency'], summary['recency'], summary['T'],
    summary['monetary_value'], time=12, freq='D', discount_rate=0.01/30
)

first = df.sort_values('order_date').groupby('customer_id').first()[['channel', 'market']]
summary = summary.join(first)

print(summary[['frequency', 'recency', 'T', 'monetary_value', 'clv_12m']].describe().round(2))

## LTV Distribution and Concentration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(summary['clv_12m'], bins=40, kde=True, ax=axes[0], color='#2E86AB')
axes[0].set_title('12-month CLV Distribution')
axes[0].set_xlabel('Expected CLV (€)')
axes[0].set_ylabel('Customers')

sorted_clv = summary.sort_values('clv_12m', ascending=False)
cumsum = sorted_clv['clv_12m'].cumsum() / sorted_clv['clv_12m'].sum()
axes[1].plot(np.arange(1, len(cumsum)+1) / len(cumsum) * 100, cumsum * 100, color='#A23B72', lw=2)
axes[1].axhline(80, color='gray', ls='--', alpha=0.7)
axes[1].set_title('CLV Concentration (Pareto)')
axes[1].set_xlabel('% of Customers')
axes[1].set_ylabel('Cumulative % of CLV')
axes[1].set_xlim(0, 100)
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()

top10_share = summary.nlargest(int(0.1 * len(summary)), 'clv_12m')['clv_12m'].sum() / summary['clv_12m'].sum()
print(f'Top 10% of customers contribute {top10_share*100:.1f}% of total 12m CLV')

## Next-Euro Ranking by Acquisition Channel

In [ ]:
channel_ltv = (
    summary.groupby('channel')['clv_12m']
    .agg(['count', 'mean', 'median', 'sum'])
    .sort_values('mean', ascending=False)
    .round(2)
)

print(channel_ltv)

fig, ax = plt.subplots(figsize=(10, 5))
channel_ltv['mean'].plot(kind='barh', color='#2E86AB', ax=ax)
ax.set_title('Mean 12-month CLV by Acquisition Channel (Next-Euro Proxy)')
ax.set_xlabel('Mean Expected CLV (€)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Top Customers and Channel Mix

In [ ]:
print('Top 10 customers by predicted 12m CLV:')
display(summary.nlargest(10, 'clv_12m')[['clv_12m', 'frequency', 'monetary_value', 'channel', 'market']].round(2))

fig, ax = plt.subplots(figsize=(8, 5))
summary.groupby('channel').size().sort_values().plot(kind='barh', color='#F18F01', ax=ax)
ax.set_title('Customers by Acquisition Channel')
ax.set_xlabel('Number of Customers')
plt.tight_layout()
plt.show()

## Recommended Actions
1. Protect and upsell high-CLV multi-category repeat buyers.
2. Use early signals (second purchase within 30 days) as strong LTV predictors.
3. Re-allocate test budget toward channels with higher mean CLV after geo/holdout validation.
4. Build self-serve cohort + LTV dashboards so Marketing and Finance see the same numbers.